# Video Game Rating Prediction - Comprehensive ML Project
## Data Preparation, EDA, Feature Engineering & Model Development

This notebook demonstrates a complete machine learning workflow including:
- **Data Cleaning & Wrangling**: Handling messy data with inconsistent formats, missing values, and nested structures
- **Exploratory Data Analysis (EDA)**: Statistical analysis and visualizations with interpretations
- **Feature Engineering**: Creating new features and transforming existing ones
- **Model Development**: Building and comparing multiple algorithms
- **Hyperparameter Tuning**: Grid search and cross-validation

**Objective**: Predict game ratings based on game characteristics (classification: High/Low rated or regression: numerical rating)

## 1. LOAD AND INSPECT THE DATASET

In [ ]:
# ============================================================================
# 1. LOAD AND INSPECT THE DATASET
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('games.csv', index_col=0)

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check for missing values and data quality issues
print("\n" + "=" * 80)
print("DATA QUALITY ASSESSMENT")
print("=" * 80)
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nMissing values percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

# Inspect sample data for specific columns
print("\n" + "-" * 80)
print("SAMPLE DATA - Showing potential data quality issues:")
print("-" * 80)
print("\nSample Release Dates (messy format):")
print(df['Release Date'].head(10))
print("\nSample Team data (nested lists):")
print(df['Team'].head(5))
print("\nSample Genres data (nested lists):")
print(df['Genres'].head(5))
print("\nSample Rating values:")
print(df['Rating'].head(10))
print(f"Rating range: {df['Rating'].min()} - {df['Rating'].max()}")

## 2. DATA CLEANING & PREPROCESSING

### Issues Identified:
1. **Release Date**: Inconsistent formats (e.g., "Feb 25, 2022") - need to parse and extract year
2. **Team, Genres, Reviews**: Stored as string representations of lists - need to parse
3. **Numeric columns**: Some values contain 'K' suffix (e.g., "3.9K") - need to convert
4. **Missing values**: Need to handle appropriately
5. **Text data**: Summary and Reviews columns contain unstructured text

In [ ]:
import ast
from dateutil.parser import parse as parse_date
from sklearn.preprocessing import LabelEncoder

# Create a copy for cleaning
df_clean = df.copy()

print("=" * 80)
print("STEP 1: PARSE NESTED LIST COLUMNS (Team, Genres)")
print("=" * 80)

def parse_list_column(value):
    """Parse string representation of list"""
    if pd.isna(value):
        return []
    try:
        return ast.literal_eval(value)
    except:
        return []

# Parse Team and Genres
df_clean['Team'] = df_clean['Team'].apply(parse_list_column)
df_clean['Genres'] = df_clean['Genres'].apply(parse_list_column)

print("\nParsed Team and Genres columns")
print(f"Sample Team: {df_clean['Team'].iloc[0]}")
print(f"Sample Genres: {df_clean['Genres'].iloc[0]}")

# ============================================================================
# STEP 2: CONVERT NUMERIC COLUMNS WITH 'K' SUFFIX
# ============================================================================

print("\n" + "=" * 80)
print("STEP 2: CONVERT NUMERIC COLUMNS (Remove 'K' suffix and convert)")
print("=" * 80)

def convert_k_to_numeric(value):
    """Convert values like '3.9K' to 3900"""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return value
    
    value = str(value).strip()
    if 'K' in value:
        return float(value.replace('K', '')) * 1000
    try:
        return float(value)
    except:
        return np.nan

# Apply conversion to numeric columns
numeric_cols = ['Times Listed', 'Number of Reviews', 'Plays', 'Playing', 'Backlogs', 'Wishlist']
for col in numeric_cols:
    df_clean[col] = df_clean[col].apply(convert_k_to_numeric)

print("Converted numeric columns:")
print(df_clean[numeric_cols].head())

In [ ]:
# ============================================================================
# STEP 3: CLEAN RELEASE DATE - Extract Year and Parse to Datetime
# ============================================================================

print("\n" + "=" * 80)
print("STEP 3: PARSE RELEASE DATE")
print("=" * 80)

def extract_year_from_date(date_str):
    """Extract year from date string"""
    if pd.isna(date_str):
        return np.nan
    try:
        # Remove quotes if present
        date_str = str(date_str).strip('"')
        parsed_date = parse_date(date_str)
        return parsed_date.year
    except:
        return np.nan

df_clean['Release_Year'] = df_clean['Release Date'].apply(extract_year_from_date)

# Fill missing years with median
median_year = int(df_clean['Release_Year'].median())
df_clean['Release_Year'] = df_clean['Release_Year'].fillna(median_year).astype('Int64')

print(f"Release Year range: {df_clean['Release_Year'].min()} - {df_clean['Release_Year'].max()}")
print(f"Filled {df_clean['Release_Year'].isna().sum()} missing values with median year {median_year}")
print(f"\nRelease Year distribution:\n{df_clean['Release_Year'].value_counts().sort_index()}")

# ============================================================================
# STEP 4: FEATURE ENGINEERING FROM CATEGORICAL COLUMNS
# ============================================================================

print("\n" + "=" * 80)
print("STEP 4: FEATURE ENGINEERING FROM TEAM AND GENRES")
print("=" * 80)

# Number of developers
df_clean['Num_Developers'] = df_clean['Team'].apply(len)
print(f"\nNumber of Developers: min={df_clean['Num_Developers'].min()}, "
      f"max={df_clean['Num_Developers'].max()}, mean={df_clean['Num_Developers'].mean():.2f}")

# Number of genres
df_clean['Num_Genres'] = df_clean['Genres'].apply(len)
print(f"Number of Genres: min={df_clean['Num_Genres'].min()}, "
      f"max={df_clean['Num_Genres'].max()}, mean={df_clean['Num_Genres'].mean():.2f}")

# Get all unique genres
all_genres = set()
for genres_list in df_clean['Genres']:
    all_genres.update(genres_list)
all_genres = sorted(list(all_genres))

print(f"\nTotal unique genres: {len(all_genres)}")
print(f"Genres: {all_genres}")

# Create binary columns for top genres
top_genres = ['Adventure', 'RPG', 'Indie', 'Platform', 'Brawler', 'Puzzle']
for genre in top_genres:
    df_clean[f'Genre_{genre}'] = df_clean['Genres'].apply(lambda x: 1 if genre in x else 0)

print(f"\nGenre distribution:")
for genre in top_genres:
    count = df_clean[f'Genre_{genre}'].sum()
    print(f"  {genre}: {count} games ({count/len(df_clean)*100:.1f}%)")

In [ ]:
# ============================================================================
# STEP 5: HANDLE MISSING VALUES
# ============================================================================

print("\n" + "=" * 80)
print("STEP 5: HANDLE MISSING VALUES")
print("=" * 80)

print("\nMissing values before imputation:")
print(df_clean.isnull().sum())

# Fill missing values
df_clean['Times Listed'].fillna(df_clean['Times Listed'].median(), inplace=True)
df_clean['Number of Reviews'].fillna(df_clean['Number of Reviews'].median(), inplace=True)
df_clean['Plays'].fillna(df_clean['Plays'].median(), inplace=True)
df_clean['Playing'].fillna(df_clean['Playing'].median(), inplace=True)
df_clean['Backlogs'].fillna(df_clean['Backlogs'].median(), inplace=True)
df_clean['Wishlist'].fillna(df_clean['Wishlist'].median(), inplace=True)

# Rating - fill with median (very few are missing)
df_clean['Rating'].fillna(df_clean['Rating'].median(), inplace=True)

print("\nMissing values after imputation:")
print(df_clean.isnull().sum())

# ============================================================================
# STEP 6: CREATE TARGET VARIABLE FOR CLASSIFICATION
# ============================================================================

print("\n" + "=" * 80)
print("STEP 6: CREATE TARGET VARIABLES")
print("=" * 80)

# Regression target: Rating (continuous)
print(f"\nRating Statistics (Regression Target):")
print(df_clean['Rating'].describe())

# Classification target: High vs Low rated games (Rating >= 4.0)
df_clean['High_Rated'] = (df_clean['Rating'] >= 4.0).astype(int)
print(f"\nClassification Target (High_Rated):")
print(f"  High rated (>=4.0): {(df_clean['High_Rated'] == 1).sum()} games ({(df_clean['High_Rated'] == 1).sum()/len(df_clean)*100:.1f}%)")
print(f"  Low rated (<4.0): {(df_clean['High_Rated'] == 0).sum()} games ({(df_clean['High_Rated'] == 0).sum()/len(df_clean)*100:.1f}%)")

print("\n✓ Data cleaning and preprocessing complete!")